# DACIL-WESENSE Quantitative Analysis Pipeline

**Pipeline overview:**

1. Environment setup and batch folder discovery
2. Data loading: CSV telemetry
3. Data synchronisation and cleaning
4. Exploratory Data Analysis (EDA)
5. Unsupervised Machine Learning (PCA, K-Means, DBSCAN)
6. Per-patient output export
7. Master summary CSV and notebook export


## 1. Environment Setup


In [1]:
if False:
    !pip install -r requirements.txt

In [2]:
import concurrent.futures
import logging
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import functions as fn
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Configure logging: messages appear both in the notebook and in a log file.
fn.setup_logging(log_file="pipeline.log", level=logging.INFO)
logger = logging.getLogger("main")

logger.info("Pipeline started.")


2026-03-04T21:06:12 | INFO     | main | Pipeline started.


## 2. Configuration

Set `DATA_ROOT` to the directory that contains one sub-folder per patient/trial.
All outputs are written under `OUTPUT_ROOT`.


In [3]:
# ----------------------------------------------------------------
# CONFIGURE THESE PATHS BEFORE RUNNING
# ----------------------------------------------------------------
DATA_ROOT: str = "data"          # root directory with patient sub-folders
OUTPUT_ROOT: str = "output"      # base output directory
# ----------------------------------------------------------------

output_base = Path(OUTPUT_ROOT)
output_base.mkdir(parents=True, exist_ok=True)

logger.info("Data root: %s", DATA_ROOT)
logger.info("Output root: %s", OUTPUT_ROOT)

# ----------------------------------------------------------------
# PERFORMANCE TOGGLES
# ----------------------------------------------------------------
USE_GPU: bool = False   # requires cupy + CUDA; falls back to CPU if unavailable
USE_PARALLEL: bool = False  # process patients concurrently
N_WORKERS: int = 4      # max parallel workers; increase up to os.cpu_count()
# ----------------------------------------------------------------

fn.configure(use_gpu=USE_GPU)


2026-03-04T21:06:12 | INFO     | main | Data root: data
2026-03-04T21:06:12 | INFO     | main | Output root: output


## 3. Batch Folder Discovery

All immediate sub-directories of `DATA_ROOT` are treated as independent
patient/trial folders.  A `try-except` block around each folder ensures
that a single problematic folder does not abort the entire batch.


In [4]:
patient_folders = fn.discover_patient_folders(DATA_ROOT)
print(f"Found {len(patient_folders)} patient folder(s).")
for pf in patient_folders:
    print(" -", pf.name)

2026-03-04T21:06:12 | INFO     | functions | Discovered 1 patient folder(s) under 'data'.


Found 1 patient folder(s).
 - Patient 1


## 4. Batch Processing Loop

For each patient folder the pipeline:

1. Loads CSV telemetry and patient metadata.
2. Saves the cleaned telemetry as a CSV.
3. Appends a summary row to the master list.

Any folder that raises an exception is logged and skipped gracefully.


In [5]:
master_records = []
processed_telemetry: dict = {}  # patient_id -> cleaned telemetry DataFrame


def _collect_result(result):
    """Append a successful process_patient result to the accumulators."""
    if result is not None:
        master_records.append(result["summary_row"])
        processed_telemetry[result["patient_id"]] = result["telemetry_df"]


if USE_PARALLEL:
    with concurrent.futures.ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(fn.process_patient, folder, output_base, skip_ecg=True): folder
            for folder in patient_folders
        }
        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=len(patient_folders),
            desc="Processing patients",
        ):
            _collect_result(future.result())
else:
    for folder in tqdm(patient_folders, desc="Processing patients"):
        _collect_result(fn.process_patient(folder, output_base, skip_ecg=True))


print(f" Batch complete. Successfully processed {len(master_records)} patient(s).")


Processing patients:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-04T21:06:12 | INFO     | functions | Processing patient: Patient 1
2026-03-04T21:06:12 | INFO     | functions |   Patient 1 — telemetry: 476 rows × 19 cols
2026-03-04T21:06:12 | INFO     | functions | Loaded BDF: WESENSETEST_04_L1_ECG_20260109T150411_AM.bdf  (8997.0 s)
2026-03-04T21:06:13 | INFO     | functions | ECG checkpoint saved: output/Patient 1/ecg_checkpoint_L1.pkl
2026-03-04T21:06:13 | INFO     | functions | Loaded BDF: WESENSETEST_04_L2_ECG_20260109T150520_AM.bdf  (9010.0 s)
2026-03-04T21:06:13 | INFO     | functions | ECG checkpoint saved: output/Patient 1/ecg_checkpoint_L2.pkl
2026-03-04T21:06:13 | INFO     | functions | Saved telemetry CSV: output/Patient 1/Patient 1_telemetry.csv
2026-03-04T21:06:13 | INFO     | functions |   Patient 1 — done.


 Batch complete. Successfully processed 1 patient(s).


## 5. Master Summary DataFrame

Compile all per-patient summary rows into a single master DataFrame and
save it as `output/master_summary.csv`.


In [6]:
if master_records:
    summary_df = pd.DataFrame(master_records)
    master_csv = output_base / "master_summary.csv"
    summary_df.to_csv(master_csv, index=False)
    logger.info("Saved master summary: %s", master_csv)
    display(summary_df.head())
else:
    summary_df = pd.DataFrame()
    print("No patient data available. Check that DATA_ROOT contains valid folders.")


2026-03-04T21:06:13 | INFO     | main | Saved master summary: output/master_summary.csv


,patient_id,peak_RER,mean_RER,peak_EqCO2,mean_EqCO2,peak_EqO2,mean_EqO2,peak_ecg_mean_abs_uV,mean_ecg_mean_abs_uV,peak_ecg_estimated_hr_bpm,mean_ecg_estimated_hr_bpm,ecg_mean_abs_uV,ecg_estimated_hr_bpm
0,Patient 1,1.34,0.828056,69.2,35.601389,56.6,29.934954,6686.87806,6686.87806,71.094793,71.094793,6686.87806,71.094793


## 6. Exploratory Data Analysis

### 6a. Per-patient stage profiles

For each successfully processed patient we plot how the key physiological
metrics (VO2, HR, Power, VCO2) evolve across the five trial stages.
Figures are saved under `output/<patient_id>/`.


In [7]:
EDA_METRICS = ["HR", "VO2", "VCO2", "Power", "RER", "SpO2"]

for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="EDA plots"):
    patient_out = output_base / patient_id
    fig = fn.plot_metrics_by_stage(
        telemetry_df,
        metrics=EDA_METRICS,
        patient_id=patient_id,
        output_dir=patient_out,
    )
    plt.show()
    plt.close(fig)


EDA plots:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-04T21:06:13 | INFO     | matplotlib.category | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-03-04T21:06:13 | INFO     | matplotlib.category | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-03-04T21:06:13 | INFO     | matplotlib.category | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-03-04T21:06:13 | INFO     | matplotlib.category | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-03-04T21:06:13 

TypeError: 'value' must be an instance of str or bytes, not a float

### 6b. Stage-level summary statistics

Mean and standard deviation of key metrics for each trial stage.


In [8]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="Stage summaries"):
    print(f"\n--- Patient: {patient_id} ---")
    stage_summary = fn.compute_stage_summary(telemetry_df, metrics=EDA_METRICS)
    display(stage_summary)


Stage summaries: 0it [00:00, ?it/s]

### 6d. Correlation heatmap

Pearson correlations between the key CPET metrics pooled across all patients.
Strong positive correlations (e.g. VO2–VCO2) confirm physiological coupling;
the RER column reveals how metabolic shift relates to ventilatory demand.


In [10]:
CORR_CANDIDATES = [
    "HR", "VO2", "VCO2", "RER", "VE", "BF", "VTex",
    "SpO2", "VO2/HR", "PETCO2", "PETO2", "EqO2", "EqCO2",
]

if processed_telemetry:
    all_tele_corr = pd.concat(processed_telemetry.values(), ignore_index=True)
    present = [fn._find_column(all_tele_corr, [c]) for c in CORR_CANDIDATES]
    present = [c for c in present if c is not None]
    corr_df = all_tele_corr[present].apply(pd.to_numeric, errors="coerce").dropna(how="all")

    fig, ax = plt.subplots(figsize=(10, 8))
    import seaborn as sns
    sns.heatmap(
        corr_df.corr(),
        annot=True, fmt=".2f", cmap="RdBu_r", center=0,
        square=True, linewidths=0.5, ax=ax,
    )
    ax.set_title("Pearson correlation – pooled CPET metrics")
    fig.tight_layout()
    fn._save_figure(fig, output_base, "corr_heatmap.png")
    plt.show()
    plt.close(fig)
else:
    print("No telemetry available for correlation analysis.")


No telemetry available for correlation analysis.


### 6e. V-slope analysis (VCO2 vs VO2)

The V-slope method plots VCO₂ against VO₂ during incremental exercise.
Below the first ventilatory threshold (VT1 / AT) the relationship is roughly linear
with slope ≈ 1 (RER < 1).  Above VT1 excess CO₂ from bicarbonate buffering causes
VCO₂ to rise more steeply.  A 45° reference line (slope = 1) is shown in grey.


In [11]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="VE/VCO2 plots"):
    vo2_col = fn._find_column(telemetry_df, ["VO2", "V'O2", "vo2"])
    vco2_col = fn._find_column(telemetry_df, ["VCO2", "V'CO2", "vco2"])
    stage_col = fn._find_column(telemetry_df, ["Stage"])
    if vo2_col is None or vco2_col is None:
        print(f"{patient_id}: VO2 or VCO2 column not found, skipping V-slope.")
        continue

    tdf = telemetry_df[[vo2_col, vco2_col]]
    if stage_col:
        tdf = telemetry_df[[vo2_col, vco2_col, stage_col]]
    tdf = tdf.apply(lambda s: pd.to_numeric(s, errors="coerce") if s.name != stage_col else s)
    tdf = tdf.dropna(subset=[vo2_col, vco2_col])

    fig, ax = plt.subplots(figsize=(6, 5))
    if stage_col and stage_col in tdf.columns:
        import seaborn as sns
        palette = dict(zip(
            ["Opwarmen", "Test", "VT1", "VT2", "Herstel"],
            sns.color_palette("tab10", 5),
        ))
        for stage, grp in tdf.groupby(stage_col):
            ax.scatter(grp[vo2_col], grp[vco2_col],
                       label=stage, color=palette.get(stage, "grey"), s=15, alpha=0.7)
    else:
        ax.scatter(tdf[vo2_col], tdf[vco2_col], s=15, alpha=0.7)

    # 45° reference line
    lim_min = min(tdf[vo2_col].min(), tdf[vco2_col].min())
    lim_max = max(tdf[vo2_col].max(), tdf[vco2_col].max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            color="grey", linestyle="--", linewidth=1, label="slope = 1")

    ax.set_xlabel(f"{vo2_col} (mL/min)")
    ax.set_ylabel(f"{vco2_col} (mL/min)")
    ax.set_title(f"{patient_id} – V-slope (VCO₂ vs VO₂)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    patient_out = output_base / patient_id
    patient_out.mkdir(parents=True, exist_ok=True)
    fn._save_figure(fig, patient_out, "vslope.png")
    plt.show()
    plt.close(fig)


VE/VCO2 plots: 0it [00:00, ?it/s]

### 6f. RER trajectory with VT2 marker

The Respiratory Exchange Ratio (RER = VCO₂/VO₂) rises progressively with exercise
intensity.  RER ≥ 1.0 is the conventional marker for VT2 (respiratory compensation
threshold).  A 5-point rolling mean smooths breath-by-breath noise.


In [12]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="RER plots"):
    rer_col = fn._find_column(telemetry_df, ["RER", "rer"])
    if rer_col is None:
        print(f"{patient_id}: RER column not found, skipping.")
        continue

    rer = pd.to_numeric(telemetry_df[rer_col], errors="coerce").dropna()
    if rer.empty:
        continue
    rer_smooth = rer.rolling(5, min_periods=1, center=True).mean()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(rer_smooth.values, label="RER (5-pt rolling mean)", linewidth=1.5)
    ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="RER = 1.0 (VT2)")

    # Annotate first crossing of RER = 1.0
    crossings = rer_smooth[rer_smooth >= 1.0]
    if not crossings.empty:
        first_idx = crossings.index[0]
        pos = rer_smooth.index.get_loc(first_idx)
        ax.axvline(pos, color="orange", linestyle=":", linewidth=1)
        ax.annotate(
            f"VT2 ≈ sample {pos}",
            xy=(pos, 1.0), xytext=(pos + 2, rer_smooth.max() * 0.95),
            arrowprops=dict(arrowstyle="->", color="orange"),
            fontsize=8, color="orange",
        )

    ax.set_xlabel("Sample index")
    ax.set_ylabel("RER")
    ax.set_title(f"{patient_id} – RER trajectory")
    ax.legend(fontsize=8)
    fig.tight_layout()
    patient_out = output_base / patient_id
    patient_out.mkdir(parents=True, exist_ok=True)
    fn._save_figure(fig, patient_out, "rer_trajectory.png")
    plt.show()
    plt.close(fig)


RER plots: 0it [00:00, ?it/s]

### 6g. Heart-rate recovery (HRR)

Heart-rate recovery is the drop in HR from peak exercise to 1 min (HRR-1) and
2 min (HRR-2) into the recovery (Herstel) stage.  Impaired HRR (< 12 bpm at 1 min)
is associated with increased cardiovascular risk.


In [13]:
hrr_records = []

for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="HRR analysis"):
    hr_col = fn._find_column(telemetry_df, ["HR", "hr", "Heart Rate"])
    stage_col = fn._find_column(telemetry_df, ["Stage"])
    if hr_col is None or stage_col is None:
        continue

    hr = pd.to_numeric(telemetry_df[hr_col], errors="coerce")
    stages = telemetry_df[stage_col]

    # Peak HR during Test / VT1 / VT2 stages
    exercise_mask = stages.isin(["Test", "VT1", "VT2"])
    if exercise_mask.sum() == 0:
        exercise_mask = ~stages.isin(["Opwarmen", "Herstel"])
    peak_hr = hr[exercise_mask].max()

    # Herstel rows in order
    herstel_hr = hr[stages == "Herstel"].dropna().reset_index(drop=True)
    if herstel_hr.empty or np.isnan(peak_hr):
        continue

    # Approximate 1 min and 2 min: data is ~5-second epochs → 12 and 24 samples
    SAMPLES_PER_MIN = 12
    hr_1min = herstel_hr.iloc[min(SAMPLES_PER_MIN - 1, len(herstel_hr) - 1)]
    hr_2min = herstel_hr.iloc[min(2 * SAMPLES_PER_MIN - 1, len(herstel_hr) - 1)]
    hrr_1 = peak_hr - hr_1min
    hrr_2 = peak_hr - hr_2min
    hrr_records.append({
        "patient_id": patient_id,
        "peak_HR": round(peak_hr, 1),
        "HRR-1 (bpm)": round(hrr_1, 1),
        "HRR-2 (bpm)": round(hrr_2, 1),
    })

if hrr_records:
    hrr_df = pd.DataFrame(hrr_records).set_index("patient_id")
    display(hrr_df)

    fig, ax = plt.subplots(figsize=(max(4, len(hrr_records) * 2), 4))
    x = np.arange(len(hrr_df))
    w = 0.35
    ax.bar(x - w / 2, hrr_df["HRR-1 (bpm)"], w, label="HRR-1 (1 min)")
    ax.bar(x + w / 2, hrr_df["HRR-2 (bpm)"], w, label="HRR-2 (2 min)")
    ax.axhline(12, color="red", linestyle="--", linewidth=1, label="HRR-1 threshold (12 bpm)")
    ax.set_xticks(x)
    ax.set_xticklabels(hrr_df.index, rotation=30, ha="right")
    ax.set_ylabel("ΔHR (bpm)")
    ax.set_title("Heart-rate recovery (HRR-1 and HRR-2)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fn._save_figure(fig, output_base, "hrr_bar.png")
    plt.show()
    plt.close(fig)
else:
    print("Insufficient data for HRR calculation (need Herstel stage and HR column).")


HRR analysis: 0it [00:00, ?it/s]

Insufficient data for HRR calculation (need Herstel stage and HR column).


### 6h. VE/VCO₂ slope (ventilatory efficiency)

The slope of the regression of minute ventilation (VE) on VCO₂ during exercise
quantifies ventilatory efficiency.  A slope < 30 is normal; a slope ≥ 34 indicates
elevated ventilatory demand as seen in heart failure and pulmonary hypertension.
Only active exercise stages (Test, VT1, VT2) are included in the regression.

In [ ]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="VE/VCO2 slope"):
    patient_out = output_base / patient_id
    fig = fn.plot_ve_vco2_slope(telemetry_df, patient_id=patient_id, output_dir=patient_out)
    if fig is not None:
        plt.show()
        plt.close(fig)


### 6i. Oxygen pulse trajectory

Oxygen pulse (VO₂/HR) is a non-invasive proxy for stroke volume × O₂ extraction.
A plateau during increasing work rate suggests inadequate cardiac output augmentation.
If a dedicated VO₂/HR column is absent it is computed from VO₂ ÷ HR.

In [ ]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="O2 pulse"):
    patient_out = output_base / patient_id
    fig = fn.plot_oxygen_pulse(telemetry_df, patient_id=patient_id, output_dir=patient_out)
    if fig is not None:
        plt.show()
        plt.close(fig)


### 6j. Ventilatory efficiency curves (VE/VO₂ and VE/VCO₂)

Both ventilatory equivalents are plotted on the same axes (5-point rolling mean).
Their nadir values are annotated.  The nadir of VE/VCO₂ is the standard ventilatory
efficiency index (normal < 30).  The curves cross near the first ventilatory threshold (VT1).

In [ ]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="Vent efficiency"):
    patient_out = output_base / patient_id
    fig = fn.plot_ventilatory_efficiency(
        telemetry_df, patient_id=patient_id, output_dir=patient_out
    )
    if fig is not None:
        plt.show()
        plt.close(fig)


### 6k. Breathing reserve summary

Breathing reserve (BR) = MVV – peak VE.  MVV is estimated as peak VE / 0.70
(assumes 30 % reserve at maximum; accepted approximation when spirometry is
unavailable).  BR < 15 % of MVV suggests a ventilatory limitation to exercise.

In [ ]:
br_records = []
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="Breathing reserve"):
    br = fn.compute_breathing_reserve(telemetry_df)
    br["patient_id"] = patient_id
    br_records.append(br)

if br_records:
    br_df = pd.DataFrame(br_records).set_index("patient_id")
    br_df["BR_flag"] = br_df["BR_pct"].apply(
        lambda v: "ventilatory limitation" if (not np.isnan(v) and v < 15) else "normal"
    )
    display(br_df)
else:
    print("No breathing reserve data available.")


### 6l. Batch CPET comparisons

Cross-patient scatter plots: peak VO₂ vs age (expected age-related decline)
and peak VO₂ vs peak power output (work-rate efficiency).

In [ ]:
if not summary_df.empty:
    batch_figs = fn.plot_batch_aggregates(summary_df, output_dir=output_base)
    for fig in batch_figs:
        plt.show()
        plt.close(fig)
else:
    print("No summary data available for batch CPET comparisons.")


## 7. Unsupervised Machine Learning

**PCA** is applied to reduce the high-dimensional telemetry feature space to
a small number of interpretable axes.  The Scree plot guides the choice of
how many components capture the majority of variance (target: >= 90 %).

**K-Means** is then applied to the original scaled space.  K-Means is chosen
because the expected physiological response groups (e.g., low/moderate/high
exertion) are approximately convex and of similar size - well-suited to
centroid-based clustering.  The Elbow plot is used to select K.

**DBSCAN** is applied as a complementary density-based algorithm, which can
identify irregularly shaped clusters and flag outlier observations as noise
(label = -1).  This helps reveal atypical physiological responses that
K-Means would force into a cluster.

Results are visualised in the PCA reduced space and as VO2 vs HR scatter
plots coloured by cluster assignment.


### 7b. Feature preparation


In [14]:
# Concatenate all patient telemetry for a global ML analysis.
if processed_telemetry:
    all_tele = pd.concat(
        [df.assign(patient_id=pid) for pid, df in processed_telemetry.items()],
        ignore_index=True,
    )

    X_scaled, feature_df, scaler = fn.prepare_features(
        all_tele, drop_cols=["patient_id"]
    )
    stage_labels = all_tele["Stage"].fillna("Unknown").reset_index(drop=True)

    print(f"Feature matrix shape: {X_scaled.shape}")
else:
    print("No telemetry data available. Skipping ML section.")
    X_scaled = None


No telemetry data available. Skipping ML section.


### 7c. PCA


In [15]:
if X_scaled is not None and X_scaled.shape[0] > 1:
    pca_model, X_pca = fn.run_pca(X_scaled, n_components=min(10, X_scaled.shape[1]))

    # Scree plot
    fig_scree = fn.plot_scree(pca_model, output_dir=output_base)
    plt.show()
    plt.close(fig_scree)

    # 2-D PCA scatter coloured by trial stage
    fig_pca_2d = fn.plot_pca_scatter(
        X_pca, stage_labels, label_name="Stage", output_dir=output_base
    )
    plt.show()
    plt.close(fig_pca_2d)

    # 3-D PCA scatter coloured by trial stage (when >= 3 PCs available)
    if X_pca.shape[1] >= 3:
        fig_pca_3d = fn.plot_pca_scatter(
            X_pca,
            stage_labels,
            label_name="Stage",
            output_dir=output_base,
            three_d=True,
        )
        plt.show()
        plt.close(fig_pca_3d)
else:
    print("Insufficient data for PCA.")


Insufficient data for PCA.


### 7d. Elbow plot for K-Means


In [16]:
if X_scaled is not None and X_scaled.shape[0] > 10:
    max_k = min(10, X_scaled.shape[0] - 1)
    fig_elbow = fn.elbow_plot(
        X_scaled, k_range=range(2, max_k + 1), output_dir=output_base
    )
    plt.show()
    plt.close(fig_elbow)
else:
    print("Insufficient data for elbow analysis.")


Insufficient data for elbow analysis.


### 7e. K-Means clustering

Based on the elbow plot above, select `N_CLUSTERS`.


In [17]:
N_CLUSTERS = 4  # adjust based on the elbow plot

if X_scaled is not None and X_scaled.shape[0] >= N_CLUSTERS:
    km_model, km_labels = fn.run_kmeans(X_scaled, n_clusters=N_CLUSTERS)

    # Determine the best available metric pair for visualisation.
    vo2_col = fn._find_column(feature_df, ["VO2", "vo2", "VO2_max"])
    hr_col = fn._find_column(feature_df, ["HR", "hr", "Heart Rate"])
    x_vis = vo2_col or feature_df.columns[0]
    y_vis = hr_col or feature_df.columns[min(1, len(feature_df.columns) - 1)]

    fig_km = fn.plot_cluster_scatter(
        feature_df,
        km_labels,
        x_col=x_vis,
        y_col=y_vis,
        algorithm_name="K-Means",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_km)

    # PCA scatter coloured by K-Means label
    km_series = pd.Series(km_labels.astype(str), name="KMeans_Cluster")
    fig_pca_km = fn.plot_pca_scatter(
        X_pca,
        km_series,
        label_name="K-Means Cluster",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_pca_km)
else:
    print("Insufficient data for K-Means.")


Insufficient data for K-Means.


**Cluster interpretation:** Each cluster corresponds to a distinct
physiological response regime identifiable in the VO2 vs HR space.
Clusters with high VO2 and high HR typically represent maximal or near-maximal
exertion (Test / VT2 stages), while low-VO2 / low-HR clusters correspond to
warm-up (Opwarmen) and recovery (Herstel).


### 7f. DBSCAN clustering

DBSCAN does not require specifying K in advance and naturally identifies noise
points (label = -1), making it suitable for detecting physiologically
atypical observations.


In [18]:
if X_scaled is not None and X_scaled.shape[0] >= 10:
    db_model, db_labels = fn.run_dbscan(X_scaled, eps=1.0, min_samples=5)
    n_clusters_found = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = int(np.sum(db_labels == -1))
    print(f"DBSCAN found {n_clusters_found} cluster(s) and {n_noise} noise point(s).")

    fig_db = fn.plot_cluster_scatter(
        feature_df,
        db_labels,
        x_col=x_vis,
        y_col=y_vis,
        algorithm_name="DBSCAN",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_db)
else:
    print("Insufficient data for DBSCAN.")


Insufficient data for DBSCAN.


**DBSCAN interpretation:** Points labelled -1 (noise) are physiological
observations that do not fit any dense neighbourhood and may indicate artefacts
or genuinely extreme responses.  The remaining clusters can be compared to
the K-Means assignment to assess stability of the groupings.


## 8. Output Summary

In [19]:
# List all files written to the output directory.
for path in sorted(output_base.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_base))

.gitkeep
Patient 1/Patient 1_metrics_by_stage.png
Patient 1/Patient 1_telemetry.csv
Patient 1/vslope.png
copd_risk/WESENSETEST04_copd_radar.png
copd_risk/copd_risk_bar.png
copd_risk_summary.csv
corr_heatmap.png
main_report.html
master_summary.csv


## 9. Notebook Export

Export this notebook to HTML or PDF using `nbconvert`:

**HTML (recommended - no additional LaTeX required):**

```bash
jupyter nbconvert --to html cpet.ipynb --output output/main_report.html
```

**PDF (requires LaTeX / pandoc to be installed):**

```bash
jupyter nbconvert --to pdf cpet.ipynb --output output/main_report.pdf
```

Or run programmatically from within the notebook:


In [20]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "html",
        "cpet.ipynb",
        "--output", str(output_base / "main_report.html"),
    ],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("Notebook exported to:", output_base / "main_report.html")
else:
    print("Export failed:", result.stderr)


Notebook exported to: output/main_report.html


---

**End of pipeline.**